# Model Creation and Training

This notebook documents how the regression models are trained for the thesis. It shows the train-test design, the 5-fold cross-validation setup, the fitted model families, the tuning outputs, and the final held-out evaluation workflow.

## Imports

In [ ]:
import os
import sys
import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from IPython.display import display

sns.set_theme(style="whitegrid", context="talk")

## Paths and configuration

In [ ]:
if os.path.basename(os.getcwd()) == "notebooks":
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
else:
    PROJECT_ROOT = os.getcwd()

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import ExperimentConfig
from src.evaluation import evaluate_all_models, evaluate_early_warning
from src.modeling import train_models
from src.utils import assert_no_post_initiation_features, set_global_seed

CONFIG_PATH = os.path.join(PROJECT_ROOT, "outputs", "tables", "run_config.json")
if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH, "r", encoding="utf-8") as handle:
        config_dict = json.load(handle)
    cfg = ExperimentConfig(**config_dict)
else:
    cfg = ExperimentConfig()

set_global_seed(cfg.random_state, cfg.python_hash_seed)
print(f"Project root: {PROJECT_ROOT}")
print(f"CV folds: {cfg.cv_folds}")
print(f"Test size: {cfg.test_size}")

## Load the final model-ready dataset

In [ ]:
dataset_path = os.path.join(PROJECT_ROOT, "data", "processed", "final_synthetic_dataset.csv")
if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"Could not find model-ready dataset: {dataset_path}")

dataset_df = pd.read_csv(dataset_path)
target_cols = ["cost_overrun_pct", "schedule_overrun_pct"]
X = dataset_df.drop(columns=target_cols)
y = dataset_df[target_cols]
assert_no_post_initiation_features(X)
print(f"Feature matrix shape: {X.shape}")
print(f"Target matrix shape: {y.shape}")
display(dataset_df.head())

## Validation design

In [ ]:
validation_design_df = pd.DataFrame(
    [
        {
            "n_total": len(X),
            "train_size_ratio": 1.0 - cfg.test_size,
            "test_size_ratio": cfg.test_size,
            "cv_folds_within_training": cfg.cv_folds,
            "cv_used_for_model_selection": True,
            "cv_used_for_hyperparameter_tuning": True,
        }
    ]
)
display(validation_design_df)

## Train the models

In [ ]:
# This call runs the same training logic used by the main project pipeline.
artifacts, test_predictions, y_test, X_test = train_models(X, y, cfg)
cv_results_df = artifacts.cv_scores.copy()
best_hyperparameters_df = artifacts.best_hyperparameters.copy()

print("Cross-validation results")
display(cv_results_df)
print("Best hyperparameters")
display(best_hyperparameters_df)

## Inspect the fitted pipelines

In [ ]:
pipeline_rows = []
for target_name, model_dict in artifacts.best_estimators.items():
    for model_name, pipeline in model_dict.items():
        step_names = list(pipeline.named_steps.keys()) if hasattr(pipeline, "named_steps") else []
        pipeline_rows.append(
            {
                "target": target_name,
                "model": model_name,
                "pipeline_steps": " -> ".join(step_names),
            }
        )
pipeline_df = pd.DataFrame(pipeline_rows)
display(pipeline_df)

## Held-out test evaluation

In [ ]:
metrics_df = evaluate_all_models(y_test, test_predictions, cfg)
early_warning_df = evaluate_early_warning(y_test, test_predictions, cfg)

print("Held-out regression metrics")
display(metrics_df)
print("Early-warning threshold metrics")
display(early_warning_df)

## Model selection by lowest cross-validated RMSE

In [ ]:
selected_models_df = (
    cv_results_df.sort_values(["target", "cv_rmse_mean"], ascending=[True, True])
    .groupby("target", as_index=False)
    .first()
)
display(selected_models_df)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, target_name in zip(axes, cv_results_df["target"].unique()):
    subset = cv_results_df[cv_results_df["target"] == target_name].copy()
    sns.barplot(data=subset, x="model", y="cv_rmse_mean", hue="model", legend=False, ax=ax, palette="deep")
    ax.set_title(f"CV RMSE comparison: {target_name}")
    ax.set_xlabel("Model")
    ax.set_ylabel("CV RMSE")
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

## Short summary

In [ ]:
print("This notebook shows how the thesis models are created and selected.")
print("It documents the train-test split, the 5-fold CV procedure, the tuned hyperparameters, and the held-out evaluation results.")
print("Together with the synthetic-data notebook and the model-review notebook, it gives examiners a transparent view of the full workflow.")